<a href="https://colab.research.google.com/github/Officialhimanshu710/web_rag/blob/main/web_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade langchain langchain-groq langchain-huggingface langchain-chroma langchain-community chromadb

In [ ]:

!pip install -q faiss-cpu

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from operator import itemgetter


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# loader1 = WebBaseLoader("https://en.wikipedia.org/wiki/Jats")
loader2 = WebBaseLoader("https://en.wikipedia.org/wiki/Bhagat_Singh")

docs = loader2.load()


text_spiliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    separators=["\n\n", "\n", " ", ""]
)

split = text_spiliter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(documents = split, embedding = embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k":6})


model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0 )

chat = ChatPromptTemplate([
    ("system","""you are a powerful assistant and based on the data provided answer on the basis of that, if a information is not present just say I dont know, Context:
{context}
"""),
    ("human","{question}")
])

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | chat
    | model
    | StrOutputParser()
)


In [ ]:
query = "from which community was bhagat singh?"
response = chain.invoke({"question":query})
print(response)